<div style="display:flex;gap:14px;align-items:center;flex-wrap:wrap;
 font-family:'Segoe UI',system-ui,sans-serif;font-size:13px;padding:10px 2px;
 border-bottom:2px solid #1B7A43;margin-bottom:4px;">
 <a href="https://colab.research.google.com/github/STG17-Africa/stg17-workshop/blob/main/notebooks/day1/D1_Agent_EN_open.ipynb" target="_blank"><img
  src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab"></a>
 <span style="color:#6B7B75;">Day 1 · open track</span>
 <span style="flex:1;"></span>
 <a href="./D1_Agent_FR_open.ipynb" style="color:#1B7A43;font-weight:600;
  text-decoration:none;">🌐 Français</a>
 <a href="./D1_Agent_EN.ipynb" style="color:#1B7A43;font-weight:600;
  text-decoration:none;">⇄ guided track</a>
</div>

<!-- From RAG to Agent - Where the Human Belongs · STG17 workshop · AfDB / STATAFRIC -->
<!-- GENERATED FILE — edit notebooks/_masters/d1_agent.master.ipynb instead. -->


<div style="background:linear-gradient(135deg,#0B2545 0%,#1B7A43 100%);
 border-radius:18px;padding:32px 38px;font-family:'Segoe UI',system-ui,sans-serif;margin-bottom:6px;">
 <div style="color:#F2A900;font-size:12.5px;letter-spacing:3px;font-weight:700;
  text-transform:uppercase;">African Development Bank · AU STATAFRIC · STG17 · Day 1 · 15:45</div>
 <div style="color:#fff;font-size:2em;font-weight:800;margin:10px 0 8px;line-height:1.15;">
  From RAG to agent</div>
 <div style="color:#dbe7e0;font-size:1.05em;line-height:1.55;max-width:900px;">
  This morning the model was handed passages you chose. Now it chooses which tool to call.
  One thing does not change: <b>the model asks, your code runs</b> — and everything your office
  needs to control lives in that gap.
 </div>
 <div style="color:#F2A900;font-size:13px;margin-top:14px;font-weight:600;">
  75 minutes · the core steps run with no API key at all</div>
</div>

> **Why an agent is a different risk.** An assistant that is wrong produces a wrong
> sentence, and a human reads it before it goes anywhere. An agent that is wrong
> takes a wrong **action**. The failure has already happened by the time anyone
> reads the output.


### The road through this laboratory

| # | Step | What you learn |
|---|------|----------------|
| 1 | The toolbox | What a tool is, and why `writes` is a field and not a comment |
| 2 | The protocol | How a model asks for something it cannot execute |
| 3 | **The gap** | The twenty lines where your office's policy lives |
| 4 | The loop, with no model | Every failure mode, reproducibly, with no API key |
| 5 | The loop, for real | The same code, driven by an actual model |
| 6 | The write tool | Refuse it, then approve it, and watch the disk |
| 7 | The budget | Why `max_steps` is a cost control, not a safety net |
| 8 | The audit log | What you show when someone asks what happened |

**Requirements.** The `stg17` toolkit and `scikit-learn`. **Steps 1–4 and 6–8 need
no model provider** — they exercise the part your office writes and owns. Step 5
needs one.

> This is deliberate. The loop, the gate, the error handling and the log are
> yours. The model is rented. Building yours first, and testing it without the
> rented part, is the whole methodological point of this session.


In [ ]:
# The toolkit, and the retrieval index this morning's laboratory built.
import subprocess
import sys

REPO = "https://github.com/STG17-Africa/stg17-workshop"

try:
    import stg17  # noqa: F401
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", f"git+{REPO}.git"], check=False)

import pandas as pd  # noqa: E402
from stg17 import agent, countries, env, llm, rag, ui  # noqa: E402
from stg17.i18n import T  # noqa: E402

S = env.setup({"scikit-learn": "scikit-learn>=1.3", "pandas": "pandas>=2.0"}, lang="EN")

COUNTRY_ISO3 = "CIV"
C = countries.get(COUNTRY_ISO3)
OUT = S.outputs(C.iso3, "d1_agent")

CORPUS_DIR = rag.write_sample_corpus(S.path("corpus", "sample"))
docs = rag.load_corpus(CORPUS_DIR)
index = rag.build_index(rag.chunk_documents(docs), kind="tfidf")
print(f"{C.name_en} ({C.iso3}) · {len(docs)} documents · -> {OUT}")

---
## 1 · The toolbox

A tool is a function you allow the model to ask for. Four of them here: three that
read, one that writes.

`writes` is a field on the tool, not a note in the documentation. A tool that only
reads can run without asking anyone. A tool that changes something — a file, a
record, an email, a payment — is a different kind of risk. Making that a typed
property means an office cannot forget which is which when it adds the fifth tool
six months from now.


In [ ]:
# Three read tools and one write tool. Read the signatures the model will see.
TOOLS = [
    agent.search_tool(index),
    agent.list_tool(docs),
    agent.calculator_tool(),
    agent.note_tool(OUT / "notes"),
]

print(agent.render_tools(TOOLS))

ui.result_table(
    pd.DataFrame([{"tool": t.name, "writes": t.writes,
                   "default policy": "refused" if t.writes else "runs"} for t in TOOLS]),
    caption=T("The default policy runs reads and refuses writes. You will change it in step 6.",
              "La politique par défaut exécute les lectures et refuse les écritures. "
              "Vous la changerez à l'étape 6."),
)

### Why a calculator, when the model can do arithmetic?

It cannot. A language model predicts the next token, so it predicts *digits* that
look right rather than computing them — and it is confidently wrong on numbers a
statistical office cares about.

Note the guard inside that tool: it accepts digits and operators and refuses
everything else. That is a whitelist, not a blacklist. Running `eval` on model
output with anything weaker is a remote code execution path, and "the model would
not do that" is not a security control.


In [ ]:
# The guard, demonstrated. Both of these are refused, not executed.
calc = agent.calculator_tool()
for expression in ["19.2 - 8.6", "__import__('os').system('echo hello')"]:
    print(f"{expression[:44]:<46} -> {calc.run(expression=expression)[:60]}")

---
## 2 · The protocol — how a model asks

The model has no ability to run anything. It emits text. We agree on a shape for
that text, and our code reads it.

Read the system prompt below, then look at the last rule in particular: *never
state a figure that a tool has not returned to you*. That is this morning's
refusal rule, moved into a setting where the consequence is larger.


In [ ]:
# What the model is told. The tool list is generated from TOOLS.
print(agent.SYSTEM.replace("<<TOOLS>>", agent.render_tools(TOOLS))[:1500])

In [ ]:
# Parsing is tolerant of what models actually do, and strict about the rest.
EXAMPLES = [
    '{"thought":"look it up","tool":"search_documents","args":{"query":"unemployment"}}',
    '```json\n{"tool":"calculate","args":{"expression":"2+2"}}\n```',
    'Certainly! {"answer":"8.6 per cent [search_documents]"} Let me know if...',
    'I will now search the documents for you.',
]
for text in EXAMPLES:
    try:
        action = agent.parse_action(text)
        what = f"answer: {action.answer[:34]}" if action.is_final else f"tool: {action.tool}"
        print(f"  OK      {text[:44]!r:<48} -> {what}")
    except agent.ProtocolError as exc:
        print(f"  REJECT  {text[:44]!r:<48} -> {exc}")

> **The last example is not a failure of the model.** It is a reply our code
> cannot act on, and the agent hands the error straight back so the model can
> correct itself. A system that crashes on a malformed reply will crash in
> production, because models produce them.


---
## 3 · The gap

Here is the whole of it, from `stg17/agent.py`:

```python
# The model has asked. Nothing has happened yet.
allowed = self.approve(tool, step.action.args)
step.approved = allowed
if not allowed:
    step.error = "refused by policy"
    transcript.append("TOOL RESULT: REFUSED — a human did not approve this call.")
    continue

result = tool.run(**step.action.args)
```

Four lines between the request and the execution. Every approval gate, audit
requirement and safety check your office needs goes there. There is nowhere else
it can go, and it cannot be added afterwards without changing this loop.

Three policies ship with the toolkit:

| Policy | Behaviour |
|---|---|
| `approve_reads_only` | Reads run. Writes are refused outright. **The default.** |
| `approve_interactive` | Reads run. A write asks a person, at the keyboard. |
| `approve_all` | Everything runs. Correct for a sample corpus; wrong for a record. |


---
## 4 · The loop, driven by a script

`ScriptedModel` replays fixed replies and makes no decisions. That is exactly what
makes it useful: it lets you reproduce every failure mode on demand, and it needs
no API key.

The script below contains, deliberately, one of each thing that goes wrong: a
good call, a fenced reply, a tool that does not exist, an unparsable reply, a
refused write, and finally an answer.


In [ ]:
# TODO: Run the agent with the scripted model and display its audit log
...

In [ ]:
# Did the refused write actually not happen? Check the disk, not the log.
notes = OUT / "notes"
print(T(f"note directory exists: {notes.exists()}",
        f"le répertoire de notes existe : {notes.exists()}"))

ui.key_concept(T(
    "A gate that logs a refusal and executes anyway is worse than no gate, because it "
    "produces a reassuring audit trail. Always verify the effect, not the record of it.",
    "Un contrôle qui journalise un refus et exécute quand même est pire que pas de "
    "contrôle, car il produit une trace d'audit rassurante. Vérifiez toujours l'effet, "
    "pas sa trace."))

---
## 5 · The loop, driven by an actual model

The same `Agent`, the same tools, the same gate. Only the source of the replies
changes.

If no provider is available, skip this and continue — you have already seen the
mechanism, and steps 6 to 8 do not need one.


In [ ]:
# TODO: Run the same agent against a real provider and compare the audit log
...

---
## 6 · The write tool, allowed

Now change one argument. Nothing else about the agent changes — which is the
point: the policy is a parameter, not a rewrite.

`approve_interactive` would ask you at the keyboard. In a notebook that blocks on
input, so the cell below uses `approve_all` and then checks the disk. In your
office, the interactive one is the honest default until you have a written policy
saying otherwise.


In [ ]:
# One argument changes. Watch both the log and the file system.
allowed = agent.Agent(
    TOOLS,
    model=agent.ScriptedModel([
        '{"thought":"save the finding","tool":"save_note","args":'
        '{"filename":"finding.md","text":"Youth unemployment exceeds the overall rate '
        'by 10.6 points (Labour Force Survey 2023 Q4, fictional sample corpus)."}}',
        '{"thought":"done","answer":"Saved. [save_note]"}',
    ]),
    approve=agent.approve_all,
).run(T("Save the finding as a note.", "Enregistrez le constat dans une note."))

written = sorted((OUT / "notes").glob("*")) if (OUT / "notes").exists() else []
print(T(f"files now on disk: {[p.name for p in written]}",
        f"fichiers désormais sur disque : {[p.name for p in written]}"))
if written:
    print("-" * 60)
    print(written[0].read_text(encoding="utf-8"))

---
## 7 · The budget

An agent decides how many model calls to make. Without a cap, the cost of one
question is unbounded — and a loop that cannot terminate will not terminate on its
own.

`max_steps` is not a safety net. It is a budget, and it is the only reason you can
put a price on a question before you ask it.


In [ ]:
# A model that never stops asking. The cap is what ends this.
runaway = agent.Agent(
    TOOLS, max_steps=3,
    model=agent.ScriptedModel(['{"tool":"list_documents","args":{}}'] * 20),
).run(T("Keep going forever.", "Continuez indéfiniment."), verbose=False)

print(T(f"stopped after {len(runaway.steps)} steps · reason: {runaway.stopped}",
        f"arrêté après {len(runaway.steps)} étapes · motif : {runaway.stopped}"))
print(runaway.answer)

---
## 8 · The audit log — the deliverable

An agent whose tool calls were not recorded cannot be explained afterwards, and a
statistical office cannot stand behind a process it cannot explain.

The log keeps what the model actually said, not a summary of it. When something
goes wrong three months from now, the raw reply is what tells you whether the
model asked for the wrong thing or your code did the wrong thing with a
reasonable request.


In [ ]:
# Everything from this laboratory, on disk, ready for Friday.
for name, this_run in [("agent_audit_scripted.json", run),
                       ("agent_audit_approved.json", allowed),
                       ("agent_audit_live.json", live)]:
    if this_run is None:
        continue
    agent.save_audit(this_run, OUT / name, meta={
        "country": C.iso3,
        "tools": [t.name for t in TOOLS],
        "write_tools": [t.name for t in TOOLS if t.writes],
        "policy": "approve_all" if this_run is allowed else "approve_reads_only",
        "max_steps": 8,
        "corpus": str(CORPUS_DIR),
    })
    print(f"  {name}")

run.audit().to_csv(OUT / "agent_audit.csv", index=False)
print(T(f"\nWritten to {OUT}", f"\nÉcrit dans {OUT}"))

---
### What you have

<table>
<tr><td><b>agent_audit_*.json</b></td><td>Every request the model made, whether it was
allowed, what came back, and the raw reply behind each one.</td></tr>
<tr><td><b>agent_audit.csv</b></td><td>The same log as a table, for a report.</td></tr>
<tr><td><b>notes/finding.md</b></td><td>What the agent wrote, once a policy allowed it to.</td></tr>
</table>

### The limits of what you built

- The protocol is text, not native function calling. Production systems use the
  provider's own tool API — this one is visible on purpose, and it occasionally
  needs a retry when the model wraps its JSON in prose.
- `approve_all` was used in step 6 for demonstration. It is the wrong default for
  anything touching a real record.
- The agent has four tools and one of them writes. Adding a fifth is where offices
  get into trouble: **every new tool is a new thing the model can ask for.**
- **Nothing here validates that the answer follows from the tool results.** The log
  proves what happened, not that it was right.

### Checkpoint

| Question | Answer |
|---|---|
| Where does an approval gate go? | Between the model's request and `tool.run` — nowhere else |
| Why is `writes` a field on `Tool`? | So the policy cannot forget which tools change something |
| What does `max_steps` control? | Cost. An agent decides how many calls to make |
| A refused write shows in the log — what must you also check? | The disk. A gate that logs but executes is worse than none |
| An agent is wrong. Why is that worse than an assistant being wrong? | The action already happened |


---
## Your turn

1. **Write the policy your office would actually use.** Not `approve_reads_only` —
   a function that inspects the arguments. Refuse `save_note` outside a named
   directory; refuse a search whose query mentions a respondent identifier. Policies
   are code, and that is what makes them testable.
2. **Add a fifth tool, then justify it.** Something your office genuinely needs —
   a lookup against a published table, perhaps. Then write two sentences on what a
   malicious or confused request to it could do. If you cannot write those two
   sentences, do not add the tool.
3. **Break it deliberately.** Write a `ScriptedModel` that asks for the same tool
   twenty times, or passes arguments of the wrong type, or requests a filename of
   `../../etc/passwd`. Fix what breaks. The `save_note` tool already sanitises its
   filename — read that line and decide whether you trust it.
4. **Measure the cost.** Run the live agent on five questions and record the number
   of model calls each took. That distribution, not an average, is what you budget
   against.

Tomorrow morning is prompt engineering: making a single call reliable enough that
an agent built on it is worth trusting.


---
> ### If something did not work
>
> **No model provider.** Steps 1–4 and 6–8 are the whole teaching content and need
> none. Step 5 is a demonstration that the same code accepts a real model.
>
> **The live agent looped without answering.** Raise `max_steps`, or narrow the
> task. A model asked to do three things at once often never decides it is done.
>
> **The live agent invented a figure.** Read its raw replies in the audit log. If it
> stated a number no tool returned, the system prompt was not enough for that model
> — that is a real finding, and it belongs in your Friday note.
>
> **A tool call failed with a TypeError.** The model passed an argument name that
> does not exist. The agent hands the error back so it can retry; if it never
> recovers, your tool descriptions are ambiguous.
